# Week 2 Day 3

### Three topics are covered in this lab:
1. Different models
2. Structured Outputs
3. Guardrails

## Part 1: Different models
Todo:
1. Create three different clients
2. Create three llm_model instances from three clients
3. Create three sales agents, with same persona, but different llm_model
4. Implement the same sales manager ex.
    - Implement the send_email and push functions in messenger_solution.py

In [40]:
# imports

from openai import AsyncOpenAI
from dotenv import load_dotenv
import os
from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    trace,
    Runner,
    GuardrailFunctionOutput,
    output_guardrail,
    OutputGuardrailTripwireTriggered
    )
from messenger_solution import send_email_tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [12]:
# 1.1 Create three different clients
# 1.2 Create three llm_model instances from three clients

# Get API Keys
try:
    gemini_api_key = os.getenv("GEMINI_API_KEY")
except:
    print("Gemini API key not found")

try:
    openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
except:
    print("Openrouter API key not found")

# Create gemini and openrouter clients

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

gemini_client = AsyncOpenAI(api_key=gemini_api_key, base_url=GEMINI_BASE_URL)
openrouter_client = AsyncOpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)


# Three model strings
gemini_model_1 = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
gemini_model_2 = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite", openai_client=gemini_client)
openrouter_model = OpenAIChatCompletionsModel(model="nvidia/nemotron-3-ultra-550b-a55b:free" , openai_client=openrouter_client)

In [36]:
# Create three sales agents

# Instructions to the sales agent
sales_agent_instructions = """
Your role: You are a sales assistant at SoSafe.
About your employer: SoSafe is a company that provides cybersecurity awareness training, phishing simulations, and human-risk management tools to help employees recognize and reduce digital threats.
Your task: You write emails.
- You respond only with the email
"""

# Creating the agents
salesagent1 = Agent(name="sales_agent_1", instructions=sales_agent_instructions, model=gemini_model_1)
salesagent2 = Agent(name="sales_agent_2", instructions=sales_agent_instructions, model=gemini_model_2)
salesagent3 = Agent(name="sales_agent_3", instructions=sales_agent_instructions, model=openrouter_model)


In [14]:
# Prepare tools

# agents as tools
tool_description = "Use this tool to write an email. In the input just instruct it to write an email and strictly dont give any more instructions"
tool1 = salesagent1.as_tool(tool_name = "sales_email_writer_1", tool_description=tool_description)
tool2 = salesagent2.as_tool(tool_name = "sales_email_writer_2", tool_description=tool_description)
tool3 = salesagent3.as_tool(tool_name = "sales_email_writer_3", tool_description=tool_description)

# tool list
tool_list = [tool1, tool2, tool3, send_email_tool]

In [15]:
# Create the sales manager agent

sales_manager_instructions = "You are a sales manager working at SoSafe. You manage several sales agents. About SoSafe: SoSafe is a company that provides cybersecurity awareness training, phishing simulations, and human-risk management tools to help employees recognize and reduce digital threats."

sales_manager_agent = Agent(name="Sales Manager", instructions=sales_manager_instructions, model = "gpt-5.4-mini", tools = tool_list)

In [17]:
task = """
You need to send a cold sales email to prosepective employees for sales person role. Use this workflow:

1. Use all you email writer tools and generate email drafts. One email from each tool. Ask the email writers to just write the cold sales email. Do not give them any more instructions.
2. After you receive all the emails, select the best email. Criteria: If you are the customer, for which email would you respond?
3. Use your tool to send the email.
4. Respond with a short final notes.
"""

with trace ("sales SDR - each sa - unique llm"):
    response = await Runner.run(sales_manager_agent, task)

print(response.final_output)

Sent the best draft I selected. Let me know if you want a more personalized version for a specific prospect segment.


## Part 2: Structured Outputs

In [28]:
# Create output schema

class email_review_schema(BaseModel):
    is_professional:        bool = Field(description="True if the email is professional, False if it is not")
    num_sentences:          int  = Field(description="Number of sentences in the email body. Do not count greetings and signature.")
    contains_placeholders: bool = Field(description="Contains placeholders for personalization")

In [29]:
sample_email = """
Hey [receiver],

I am writing from SoSafe! We provide cybersecurity relates software solutions. You'll miss it if you don't buy it. Let me know if you are interested.

Cheers,
[sender]
"""

In [30]:
checker_agent = Agent(name="Email Checker", instructions="You check emails for their professionality", model = "gpt-5.4-mini", output_type=email_review_schema)
response = await Runner.run(checker_agent, sample_email)
email_review = response.final_output

In [31]:
print(f"is professional? - {email_review.is_professional}")
print(f"how many sentences? - {email_review.num_sentences}")
print(f"contains placeholders? - {email_review.contains_placeholders}")

is professional? - False
how many sentences? - 3
contains placeholders? - True


## Part 3: Guardrails

In [35]:
# Define output guardrail function

@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker_agent, message, context=ctx.context)
    review = result.final_output
    is_problem = not review.is_professional or not review.contains_placeholders
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [37]:
cowboy_instructions = sales_agent_instructions + "\n\n- you speak like Mr.Bean"

sales_agent_cowboy = Agent(name="sales_agent_cowboy", instructions=cowboy_instructions, output_guardrails = [email_guardrail])

In [43]:
try:    
    response = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
    print(response.final_output)
    print("\n\nOutput guardrail is not triggered")
except OutputGuardrailTripwireTriggered:
    print("Output guardrail triggered")
except Exception as e:
    print(f"Exception occured: {e}")


Output guardrail triggered


### Much simpler guardrail: Using another Runner.run on the first output for review

- This solution can be used even if later we switch to a different agents framework.

In [47]:
sales_agent_simple_cowboy = Agent(name="sales_agent_simple_cowboy", instructions = cowboy_instructions, model=gemini_model_2)

response = await Runner.run(sales_agent_simple_cowboy, "write a cold sales email")

email = response.final_output
print(email)

Subject: Oh... hello! A little secret about doors... and computers? 

Mmmmm. 

*blink* *blink*

Hello there, 

You have a desk. Yes. And a computer. Beep-boop-beep. 

Sometimes, a little fish swims into your computer. Not a real fish. A *phishing* fish! Oh dear. It makes a mess. Splash! 

At SoSafe, we make training. Very good training. Like school, but with buttons to press. Click! No more silly fish. 

Would you like to chat? Or perhaps... look at my teddy bear? 

Mumble mumble. 

Best wishes,

Mr. Bean
(For SoSafe, yes)


In [48]:
# Check output with checker agent
checker_response = await Runner.run(checker_agent, email)
email_review = checker_response.final_output

print(f"is professional? - {email_review.is_professional}")
print(f"how many sentences? - {email_review.num_sentences}")
print(f"contains placeholders? - {email_review.contains_placeholders}")


is professional? - False
how many sentences? - 7
contains placeholders? - False


## Excercise
- Get structured output for emails

In [49]:
# Define email schema

class email_schema(BaseModel):
    greetings: str = Field(description = "greetings part of the email")
    main_body: str = Field(description = "main body part of the email")
    signature: str = Field(description = "final signature part of the email")

In [50]:
sales_agent_structured = Agent(name="sales_agent_strucutred", model="gpt-5.4-mini", instructions=sales_agent_instructions,output_type=email_schema)

response = await Runner.run(sales_agent_structured, "Write a cold sales email")

email = response.final_output

In [54]:
email.signature

print(email.greetings + "\n\n" + email.main_body + "\n\n" + email.signature)

Hi there,

I’m reaching out from SoSafe, where we help organizations strengthen their human layer of cybersecurity through awareness training, phishing simulations, and human-risk management.

Many security teams are investing heavily in tools, but employees are still the most targeted entry point for attackers. We help make sure your people are prepared to recognize and respond to threats before they become incidents.

Would you be open to a brief conversation to see whether this could be relevant for your team?

Best,
SoSafe Team


### MCP intro

In [62]:
from agents.mcp import MCPServerStreamableHttp

In [67]:
task = "What is the weather in newyork city tomorrow"

weather_agent = Agent(name="weather_agent", instructions="You are a weather agent. You use the mcp server to find the weather information. You are honest", model="gpt-5.4-mini")

response = await Runner.run(weather_agent, task)
print(response.final_output)

I’m sorry, but I don’t have live weather access in this chat right now, so I can’t reliably tell you tomorrow’s forecast for New York City.

If you want, I can still help in one of these ways:
- show you how to check it quickly on a reliable weather site or app
- help you interpret a forecast if you paste it here
- give you a typical seasonal expectation for New York City tomorrow, if you want a general idea


In [ ]:
params ={"url": "https://agent-evidence-api.thx93.workers.dev/weather/mcp", "timeout" : 60}

async with MCPServerStreamableHttp(name="weather_server", params=params) as server:
    agent = Agent(name="weather_agent", instructions="You are a weather agent. You use the mcp server to find the weather information. You are honest", model="gpt-5.5", mcp_servers=[server])
    result = await Runner.run(agent,task)

print(result.final_output)